In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path('../data')

# DISPLAY SETTINGS
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print("Setup complete ✓")

Setup complete ✓


In [3]:
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
category_translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

for name, df in {
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'customers': customers,
    'category_translation': category_translation
}.items():
    print(f"{name:25s} {df.shape}")

orders                    (99441, 8)
order_items               (112650, 7)
payments                  (103886, 5)
reviews                   (99224, 7)
products                  (32951, 9)
sellers                   (3095, 4)
customers                 (99441, 5)
category_translation      (71, 2)


In [4]:
print(orders.head())
print()
print(orders.dtypes)
print()
print(orders.isnull().sum())

                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [10]:
# Parse datetime columns
datetime_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col])

print(orders.dtypes)

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [11]:
# Engineer key features

# Only keep delivered orders for delivery analysis
delivered = orders[orders['order_status'] == 'delivered'].copy()

# Delivery delay: actual vs estimated (positive = late)
delivered['delay_days'] = (
    delivered['order_delivered_customer_date'] - 
    delivered['order_estimated_delivery_date']
).dt.days

# Binary late flag
delivered['is_late'] = (delivered['delay_days'] > 0).astype(int)

# Processing time: purchase to carrier handoff
delivered['processing_days'] = (
    delivered['order_delivered_carrier_date'] - 
    delivered['order_purchase_timestamp']
).dt.days

# Total delivery time: purchase to customer
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - 
    delivered['order_purchase_timestamp']
).dt.days

print(f"Delivered orders: {len(delivered):,}")
print(f"\nDelay statistics:")
print(delivered[['delay_days', 'is_late', 'processing_days', 'delivery_days']].describe())

Delivered orders: 96,478

Delay statistics:
       delay_days  is_late  processing_days  delivery_days
count    96470.00 96478.00         96476.00       96470.00
mean       -11.88     0.07             2.74          12.09
std         10.18     0.25             3.61           9.55
min       -147.00     0.00          -172.00           0.00
25%        -17.00     0.00             1.00           6.00
50%        -12.00     0.00             2.00          10.00
75%         -7.00     0.00             4.00          15.00
max        188.00     1.00           125.00         209.00


In [12]:
# Merge core tables

# Add order items (price, freight)
df = delivered.merge(order_items, on='order_id', how='left')

# Add reviews (satisfaction scores)
df = df.merge(
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='left'
)

# Add products (category)
df = df.merge(
    products[['product_id', 'product_category_name']],
    on='product_id',
    how='left'
)

# Add category translation (English names)
df = df.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

# Add seller state
df = df.merge(
    sellers[['seller_id', 'seller_state']],
    on='seller_id',
    how='left'
)

# Add customer state
df = df.merge(
    customers[['customer_id', 'customer_state']],
    on='customer_id',
    how='left'
)

print(f"Master dataframe shape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Master dataframe shape: (110840, 23)

Missing values:
order_approved_at                  15
order_delivered_carrier_date        2
order_delivered_customer_date       8
delay_days                          8
processing_days                     2
delivery_days                       8
review_score                      827
product_category_name            1545
product_category_name_english    1567
dtype: int64


In [13]:
# Clean master dataframe

# Drop rows with missing delivery-critical timestamps
df = df.dropna(subset=['delay_days', 'processing_days', 'delivery_days'])

# Drop rows with implausible processing times (data errors)
df = df[df['processing_days'] >= 0]

# Rename english category for convenience
df = df.rename(columns={'product_category_name_english': 'category'})

# Drop original portuguese category column
df = df.drop(columns=['product_category_name'])

print(f"Clean dataframe shape: {df.shape}")
print(f"\nRemaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Clean dataframe shape: (110645, 22)

Remaining missing values:
order_approved_at      15
review_score          825
category             1566
dtype: int64


In [14]:
# Save clean dataframe
df.to_csv(DATA_DIR / 'olist_clean.csv', index=False)

print(f"Saved olist_clean.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved olist_clean.csv
Shape: (110645, 22)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delay_days', 'is_late', 'processing_days', 'delivery_days', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'review_score', 'category', 'seller_state', 'customer_state']
